# CurvLearn — Tree-distance embedding: does hyperbolic curvature help *at all*?

The Dyck LM difficulty sweep found hyperbolic curvature **hurts**. But that used κ inside
attention scores — not the way hyperbolic geometry is supposed to help. This is the clean test:
embed the nodes of a tree as κ-stereographic points and fit **geodesic distance to tree
distance**, measuring average distortion vs κ across tree depths.

**If hyperbolic wins here** (distortion lower at κ<0, gap growing with depth) → the LM result
was about the attention mechanism, and the geometry itself is sound. **If it loses here too** →
the implementation is suspect. Either way it's decisive.

GPU runtime. Results persist to Drive; sweep is checkpointed/resumable.


## 1. GPU


In [ ]:
!nvidia-smi -L || echo 'CPU (fine here — trees are small)'

## 2. Code + install (autoreload on)


In [ ]:
%load_ext autoreload
%autoreload 2
import os
if not os.path.isdir('CurvLearn') and os.path.basename(os.getcwd()) != 'CurvLearn':
    !git clone https://github.com/TheFausap/CurvLearn.git
if os.path.basename(os.getcwd()) != 'CurvLearn':
    os.chdir('CurvLearn')
!pip -q install -e .
print('cwd:', os.getcwd())

## 3. Sanity: tree generator


In [ ]:
!python tests/test_trees.py

## 4. Drive + output folder


In [ ]:
import os, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/CurvLearn/results'
except ModuleNotFoundError:
    OUTDIR = 'results'
os.makedirs(OUTDIR, exist_ok=True)
print('device:', dev, '| results ->', OUTDIR)

## 5. Run the tree-embedding sweep
3 tree depths x 7 κ x 3 seeds = 63 embeddings, each a fast fit (~seconds). Checkpointed to
Drive. Lower `dim` (5, 2) in the spec to amplify the hyperbolic advantage if depth alone
doesn't separate the curves.


In [ ]:
import json
from curvlearn.embed_tree import run_tree_sweep, default_spec

spec = default_spec()
spec['device'] = dev
# spec['dim'] = 5        # uncomment to stress Euclidean harder
ckpt = os.path.join(OUTDIR, 'tree_embed_results.json')
out = run_tree_sweep(spec, checkpoint_path=ckpt, resume=True)
print('runs:', len(out['records']), '| saved ->', ckpt)

## 6. Plot: distortion vs curvature


In [ ]:
import json
from curvlearn import plot
from IPython.display import Image, display
out = json.load(open(os.path.join(OUTDIR, 'tree_embed_results.json')))
png = os.path.join(OUTDIR, 'tree_embed.png')
plot.tree_embed_plot(out, png)
display(Image(png))
print('saved', png)

## 7. Read it
- **Right panel dips below 0 at κ<0, deeper trees dipping more** → hyperbolic embeds trees
  with lower distortion than Euclidean, exactly as theory predicts. The Dyck LM negative
  result is then about the attention mechanism, not the geometry → next step is a better way
  to wire hyperbolic structure into the model.
- **Flat / rises at κ<0 even for deep trees** → the κ-stereographic implementation isn't
  delivering the hyperbolic advantage even on the task built for it → debug the geometry
  (try dim=2, larger |κ|, more steps) before anything else.
- If curves separate only at small `dim`, that's the expected signature: hyperbolic's edge is
  largest when the Euclidean dimension is the binding constraint.
